# Kick the Fly as a Python library

A whole-brain LIF simulation of the MaleCNS v1.0 connectome (166,700 neurons), driven and recorded from Python. `Fly` is the same headless machinery the Lab, the protocols and the validation suite use; see [api.md](api.md).

Run this from the repo root after building the brain pack (README: Run from source).

In [1]:
from kickthefly import Fly

fly = Fly(seed=1000)          # warmed up (3 s), untrained: never your saved training memory
print(fly.n, 'neurons on', fly.backend)

166700 neurons on numba


## Looming detectors onto the giant fiber

The validated pathway: drive LPLC2 + LC4 and watch DNp01, the giant fiber escape neuron.

In [2]:
rec = fly.record({'giant fiber': 'dnp01', 'looming': 'loom', 'jump DNs': 'jump'})
fly.step(2.0)                 # 2 s calm
fly.drive('loom', amp=0.5)    # like optogenetic activation
fly.step(2.0)
fly.undrive()

calm, driven = rec.rates(0, 2), rec.rates(2, 4)
for g in calm:
    print(f"{g:12s} {calm[g]:6.1f} -> {driven[g]:6.1f} spikes/s")

giant fiber     7.2 ->   66.5 spikes/s
looming         5.1 ->   67.0 spikes/s
jump DNs        2.8 ->    2.6 spikes/s


## The same stimulus with the looming detectors silenced

`silence()` is the game's brain surgery OFF. A new fly with the same seed replays the same noise.

In [3]:
lesioned = Fly(seed=1000).silence('type:LPLC2,LC4')
rec2 = lesioned.record({'giant fiber': 'dnp01'})
lesioned.step(2.0)
lesioned.drive('loom')
lesioned.step(2.0)
print('giant fiber while driving, lesioned:', round(rec2.rates(2, 4)['giant fiber'], 1), 'spikes/s')

giant fiber while driving, lesioned: 1.0 spikes/s


## Who is it?

Neurons are named as in protocols; `describe()` gives the dataset's labels.

In [4]:
for row in fly.neurons('pip10'):
    print(fly.describe(row))

{'row': 1032, 'type': 'pIP10', 'instance': 'pIP10_R', 'superclass': 'descending_neuron', 'body_id': 11116}
{'row': 132077, 'type': 'pIP10', 'instance': 'pIP10_L', 'superclass': 'descending_neuron', 'body_id': 523998}


## Strongest paths between two neurons

The big brain view's path tracer, from a script.

In [5]:
from kickthefly.lab import neurosearch

sim = fly.brain.sim
src, dst = fly.neurons('pip10')[0], fly.neurons('ps1')[0]
for p in neurosearch.top_paths(sim.W_csr, src, dst, k=3, Wc=sim.W_csc):
    print(' > '.join(fly.describe(i)['type'] for i in p['nodes']), f"{p['strength']:.1e}", '+' if p['sign'] > 0 else '-')

pIP10 > DNg74_b > ps1 MN 3.5e-05 -
pIP10 > IN08B006 > ps1 MN 1.8e-05 +
pIP10 > IN00A013 > IN19B095 > ps1 MN 1.6e-05 -


## Spikes and export

`spikes()` gives every recorded spike; `export()` writes the Lab's CSV/JSON files (and `.nwb` with `nwb=True`, which needs `pip install pynwb`).

In [6]:
t, rows = rec.spikes()
print(len(t), 'spikes; first at', t[0], 's from', fly.describe(rows[0])['type'])
files = fly.export('/tmp/kickthefly-example/looming')
print([f.name for f in files])

45252 spikes; first at 0.0 s from LPLC2
['looming-spikes.csv', 'looming-rates.csv', 'looming-group-rates.csv', 'looming.npz', 'looming-metadata.json']
